# ⚡ QAOA Green Energy Dispatch — Demo Notebook

**VTU Qiskit Fall Fest 2026 — Track 6**

This notebook walks through the complete pipeline:
1. Problem generation (QUBO formulation)
2. QUBO matrix visualisation
3. QAOA circuit construction (p=1)
4. Noiseless QAOA run
5. Noisy QAOA run (hardware noise model)
6. Classical baselines (brute force + greedy)
7. Results comparison
8. CO₂ savings visualisation


## Cell 1 — Imports & Path Setup

In [ ]:
import sys
import os

# Make src/ importable from notebooks/
_NB_DIR = os.path.abspath(".")
_PROJECT_ROOT = os.path.dirname(_NB_DIR)
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Core project modules
from src.data_generator import generate_daily_profile
from src.problem import ProblemParams, QuboMatrix
from src.classical_solvers import BruteForceSolver, GreedySolver
from src.co2_calculator import calculate_metrics, format_metrics_table
from src.noise_model import build_noise_model

print("✅ All imports successful")
print(f"   Python  : {sys.version.split()[0]}")

try:
    import qiskit
    print(f"   Qiskit  : {qiskit.__version__}")
except ImportError:
    print("   ⚠️  Qiskit not found — install with: pip install qiskit qiskit-aer")

try:
    import qiskit_aer
    print(f"   Aer     : {qiskit_aer.__version__}")
except ImportError:
    print("   ⚠️  qiskit-aer not found")


## Cell 2 — Generate Problem (4 sources × 3 time slots = 12 qubits)

In [ ]:
N_SLOTS = 3  # 4 sources × 3 slots = 12 qubits

profile = generate_daily_profile(n_slots=N_SLOTS)
print("Daily profile:")
for i, (t, d, s) in enumerate(zip(
        profile["time_labels"], profile["demands"], profile["solar_availability"])):
    print(f"  Slot {i} [{t}]  demand={d:.2f} kW  solar_avail={s:.3f}")

params = ProblemParams(
    n_slots=N_SLOTS,
    demands=profile["demands"],
    solar_avail=profile["solar_availability"],
    penalty_weight=10.0,
    co2_weight=0.5,
)

print(f"\nProblem dimensions:")
print(f"  Sources  : {params.n_sources}  ({', '.join(params.source_names)})")
print(f"  Slots    : {params.n_slots}")
print(f"  Qubits   : {params.n_vars}")

qubo = QuboMatrix(params).build()
print(f"\nQUBO matrix shape : {qubo.Q.shape}")
print(f"QUBO offset       : {qubo.offset:.4f}")


## Cell 3 — QUBO Matrix Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
Q = qubo.Q
im = ax.imshow(Q, cmap="RdBu_r", aspect="auto")
fig.colorbar(im, ax=ax, label="Q value")

# Axis labels: source_slot
tick_labels = [
    f"{src[0]}{t}"
    for src in params.source_names
    for t in range(params.n_slots)
]
ax.set_xticks(range(params.n_vars))
ax.set_yticks(range(params.n_vars))
ax.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(tick_labels, fontsize=9)
ax.set_title(f"QUBO Matrix Q  ({params.n_vars}×{params.n_vars})", fontsize=13)

# Draw grid lines at source boundaries
for k in range(params.n_sources):
    pos = k * params.n_slots - 0.5
    ax.axhline(pos, color="black", linewidth=1.5, alpha=0.4)
    ax.axvline(pos, color="black", linewidth=1.5, alpha=0.4)

plt.tight_layout()
plt.savefig("../results/qubo_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/qubo_heatmap.png")


## Cell 4 — Build and Draw QAOA Circuit (p=1)

In [ ]:
from src.qaoa_solver import QAOASolver

solver_p1 = QAOASolver(qubo, p=1, shots=1024)
circuit = solver_p1.build_circuit()

print(f"QAOA Circuit (p=1)")
print(f"  Qubits       : {circuit.num_qubits}")
print(f"  Parameters   : {len(circuit.parameters)}")
print(f"  Gates (depth): {circuit.depth()}")
print()

# Draw circuit (text mode — always works)
try:
    drawing = circuit.draw(output="text", fold=100)
    print(drawing)
except Exception:
    print(circuit)

# Also try matplotlib drawing
try:
    fig = circuit.draw(output="mpl", fold=40)
    fig.savefig("../results/qaoa_circuit_p1.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved: results/qaoa_circuit_p1.png")
except Exception as e:
    print(f"(matplotlib draw unavailable: {e})")


## Cell 5 — Run QAOA (Noiseless, Statevector Simulator)

In [ ]:
print("Running noiseless QAOA (p=1) …")
qaoa_result_clean = solver_p1.solve(use_noise=False)

print(f"\n✅ QAOA Noiseless Results")
print(f"  Best bitstring : {qaoa_result_clean['best_bitstring']}")
print(f"  Best QUBO cost : {qaoa_result_clean['best_cost']:.6f}")
print(f"  Circuit depth  : {qaoa_result_clean['circuit_depth']}")
print(f"  Runtime        : {qaoa_result_clean['runtime_s']:.2f} s")
print(f"\nSchedule:")
for source, slots in qaoa_result_clean['schedule'].items():
    icons = ["🟩" if v else "⬜" for v in slots]
    print(f"  {source:10s}: {''.join(icons)}  {slots}")

qaoa_metrics_clean = calculate_metrics(qaoa_result_clean["schedule"], params)
print(f"\nMetrics:")
print(f"  Total cost   : ${qaoa_metrics_clean['total_cost_usd']:.4f}")
print(f"  Total CO₂    :  {qaoa_metrics_clean['total_co2_kg']:.4f} kg")
print(f"  Demand met   :  {qaoa_metrics_clean['demand_satisfaction_pct']:.1f}%")


## Cell 6 — Run QAOA (Noisy Simulation)

In [ ]:
noise_model = build_noise_model(p1=0.001, p2=0.01, p_meas=0.02)
if noise_model is None:
    print("⚠️  qiskit-aer not installed — skipping noisy simulation.")
    qaoa_result_noisy = None
    qaoa_metrics_noisy = None
else:
    print("Running noisy QAOA (p=1, depolarising + readout errors) …")
    qaoa_result_noisy = solver_p1.solve_with_noise(noise_model=noise_model)

    print(f"\n✅ QAOA Noisy Results")
    print(f"  Best bitstring : {qaoa_result_noisy['best_bitstring']}")
    print(f"  Best QUBO cost : {qaoa_result_noisy['best_cost']:.6f}")
    print(f"  Runtime        : {qaoa_result_noisy['runtime_s']:.2f} s")

    qaoa_metrics_noisy = calculate_metrics(qaoa_result_noisy["schedule"], params)
    print(f"\nMetrics:")
    print(f"  Total cost   : ${qaoa_metrics_noisy['total_cost_usd']:.4f}")
    print(f"  Total CO₂    :  {qaoa_metrics_noisy['total_co2_kg']:.4f} kg")
    print(f"  Demand met   :  {qaoa_metrics_noisy['demand_satisfaction_pct']:.1f}%")

    delta_cost = qaoa_result_noisy["best_cost"] - qaoa_result_clean["best_cost"]
    print(f"\nNoise impact on QUBO cost: {delta_cost:+.6f} "
          f"({'degraded' if delta_cost > 0 else 'improved by noise'})")


## Cell 7 — Run Classical Baselines (Brute Force + Greedy)

In [ ]:
# Brute Force
print(f"Running Brute Force over 2^{params.n_vars} = {2**params.n_vars} bitstrings …")
bf = BruteForceSolver(qubo)
bf_result = bf.solve()
bf_metrics = calculate_metrics(bf_result["schedule"], params)

print(f"\n✅ Brute Force Results")
print(f"  Best bitstring : {bf_result['best_bitstring']}")
print(f"  Best QUBO cost : {bf_result['best_cost']:.6f}")
print(f"  Evaluated      : {bf_result['n_evaluated']} bitstrings")
print(f"  Runtime        : {bf_result['runtime_s']*1000:.1f} ms")
print(f"  Total cost     : ${bf_metrics['total_cost_usd']:.4f}")
print(f"  Total CO₂      : {bf_metrics['total_co2_kg']:.4f} kg")

print()
# Greedy
print("Running Greedy solver …")
gs = GreedySolver(qubo)
gs_result = gs.solve()
gs_metrics = calculate_metrics(gs_result["schedule"], params)

print(f"\n✅ Greedy Results")
print(f"  Best bitstring : {gs_result['best_bitstring']}")
print(f"  Best QUBO cost : {gs_result['best_cost']:.6f}")
print(f"  Runtime        : {gs_result['runtime_s']*1000:.3f} ms")
print(f"  Total cost     : ${gs_metrics['total_cost_usd']:.4f}")
print(f"  Total CO₂      : {gs_metrics['total_co2_kg']:.4f} kg")
print(f"  Demand met     : {gs_metrics['demand_satisfaction_pct']:.1f}%")


## Cell 8 — Results Comparison Table

In [ ]:
import pandas as pd

rows = [
    {
        "Solver": "QAOA (noiseless)",
        "QUBO Cost": round(qaoa_result_clean["best_cost"], 6),
        "Total Cost ($)": qaoa_metrics_clean["total_cost_usd"],
        "Total CO₂ (kg)": qaoa_metrics_clean["total_co2_kg"],
        "Demand Met (%)": qaoa_metrics_clean["demand_satisfaction_pct"],
        "Cost Saved ($)": qaoa_metrics_clean["cost_vs_diesel_savings_usd"],
        "CO₂ Saved (kg)": qaoa_metrics_clean["co2_vs_diesel_savings_kg"],
    },
]

if qaoa_result_noisy is not None:
    rows.append({
        "Solver": "QAOA (noisy)",
        "QUBO Cost": round(qaoa_result_noisy["best_cost"], 6),
        "Total Cost ($)": qaoa_metrics_noisy["total_cost_usd"],
        "Total CO₂ (kg)": qaoa_metrics_noisy["total_co2_kg"],
        "Demand Met (%)": qaoa_metrics_noisy["demand_satisfaction_pct"],
        "Cost Saved ($)": qaoa_metrics_noisy["cost_vs_diesel_savings_usd"],
        "CO₂ Saved (kg)": qaoa_metrics_noisy["co2_vs_diesel_savings_kg"],
    })

rows.extend([
    {
        "Solver": "Brute Force (optimal)",
        "QUBO Cost": round(bf_result["best_cost"], 6),
        "Total Cost ($)": bf_metrics["total_cost_usd"],
        "Total CO₂ (kg)": bf_metrics["total_co2_kg"],
        "Demand Met (%)": bf_metrics["demand_satisfaction_pct"],
        "Cost Saved ($)": bf_metrics["cost_vs_diesel_savings_usd"],
        "CO₂ Saved (kg)": bf_metrics["co2_vs_diesel_savings_kg"],
    },
    {
        "Solver": "Greedy",
        "QUBO Cost": round(gs_result["best_cost"], 6),
        "Total Cost ($)": gs_metrics["total_cost_usd"],
        "Total CO₂ (kg)": gs_metrics["total_co2_kg"],
        "Demand Met (%)": gs_metrics["demand_satisfaction_pct"],
        "Cost Saved ($)": gs_metrics["cost_vs_diesel_savings_usd"],
        "CO₂ Saved (kg)": gs_metrics["co2_vs_diesel_savings_kg"],
    },
])

df = pd.DataFrame(rows).set_index("Solver")
print(df.to_string())

# Compute approximation ratio
opt_cost = bf_result["best_cost"]
qaoa_cost = qaoa_result_clean["best_cost"]
approx_ratio = opt_cost / qaoa_cost if qaoa_cost != 0 else float("nan")
print(f"\n📐 QAOA Approximation Ratio (noiseless): {approx_ratio:.4f}")
print(f"   (1.0 = optimal, lower = worse)")


## Cell 9 — CO₂ Savings Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Panel 1: CO₂ per source (QAOA schedule) ---
co2_src = qaoa_metrics_clean["co2_by_source"]
colours = ["#FFD600", "#00B0FF", "#66BB6A", "#EF5350"]
labels = list(co2_src.keys())
values = list(co2_src.values())

axes[0].pie(
    [max(0, v) for v in values],
    labels=labels,
    colors=colours,
    autopct="%1.1f%%",
    startangle=140,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
)
axes[0].set_title("CO₂ by Source\n(QAOA Schedule)", fontsize=11)

# --- Panel 2: Cost comparison bar chart ---
solver_names = ["QAOA\n(noiseless)", "Brute\nForce", "Greedy"]
costs = [
    qaoa_metrics_clean["total_cost_usd"],
    bf_metrics["total_cost_usd"],
    gs_metrics["total_cost_usd"],
]
bar_colours = ["#6929C4", "#009D9A", "#FF7EB6"]
bars = axes[1].bar(solver_names, costs, color=bar_colours, edgecolor="white", width=0.5)
axes[1].set_title("Total Operational Cost", fontsize=11)
axes[1].set_ylabel("USD")
axes[1].bar_label(bars, labels=[f"${v:.4f}" for v in costs], padding=3, fontsize=8)
axes[1].set_ylim(0, max(costs) * 1.25)

# --- Panel 3: CO₂ savings vs diesel baseline ---
solvers = ["QAOA\n(noiseless)", "Brute\nForce", "Greedy"]
savings_co2 = [
    qaoa_metrics_clean["co2_vs_diesel_savings_kg"],
    bf_metrics["co2_vs_diesel_savings_kg"],
    gs_metrics["co2_vs_diesel_savings_kg"],
]
bar2 = axes[2].bar(solvers, savings_co2, color=["#6929C4", "#009D9A", "#FF7EB6"],
                   edgecolor="white", width=0.5)
axes[2].set_title("CO₂ Saved vs All-Diesel", fontsize=11)
axes[2].set_ylabel("kg CO₂ avoided")
axes[2].bar_label(bar2, labels=[f"{v:.3f} kg" for v in savings_co2], padding=3, fontsize=8)
axes[2].axhline(0, color="black", linewidth=0.8)
axes[2].set_ylim(min(savings_co2) * 1.2, max(savings_co2) * 1.3)

plt.suptitle("QAOA Green Dispatch — CO₂ & Cost Results", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../results/co2_savings.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/co2_savings.png")

print(format_metrics_table(qaoa_metrics_clean))
